# 🌌 Equatorial Plasma Bubbles (EPB) Detection System — V4

## Two-Stage Cascaded Pipeline: Astronomical Sky Quality Control + 2DPCA Stacking Ensemble

### Key Methodological & Experimental Enhancements (WAIMLAP 2026 Revision):
- ✅ **Two-Stage Cascaded Architecture**: Stage 1 Astronomical Sky Quality Control (star-count & background brightness) + Stage 2 2DPCA + Stacking Ensemble (SVM RBF + XGBoost + Random Forest → Logistic Regression meta-learner).
- ✅ **Full 2DPCA Spectrum & Cross-Validated Component Selection ($k=1\dots 30$)**: Evaluates $k=5$ ($320$ features, $59.41\%$ cumulative variance on original training split / $56.5\%$ on augmented split) across the full 64-eigenvalue spectrum.
- ✅ **Leakage-Free 5-Fold Stratified CV & Temporal-Block Group CV**: Evaluates both stratified folds and 77 non-overlapping 30-minute observation blocks to eliminate temporal autocorrelation.
- ✅ **Comprehensive Baselines, Ablations & Cross-Station Generalization**: Benchmarks against 1D-PCA, flattened classifiers, a 2D-CNN baseline, Raw Images (no physics preprocessing), No-SMOTE, and Circular FOV Masking, including Zero-Shot transfer to Cachoeira Paulista (CP).
- ✅ **Hard Negatives & Stage 1 Cloud QC**: Evaluates high-lunar-illumination frames ($\ge 50\%$), per-observatory performance, zero-shot holdout on CP, and Stage 1 cloud rejection on overcast sequences.

---
**Author:** Alberto Ivo da Costa Vieira


In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from epb_detection_system_v2 import (
    load_epb_dataset, augment_epb_data,
    optimize_n_components, plot_2dpca_variance,
    plot_performance_metrics, plot_confusion_matrix_analysis,
    plot_comprehensive_analysis, explain_feature_importance,
    analyze_threshold_sensitivity, plot_calibration_analysis,
    evaluate_kfold_cv, benchmark_baselines,
    evaluate_hard_negatives_and_generalization,
    plot_pipeline_architecture,
    explain_shap_importance, setup_logging,
    EPBRecognitionSystem, EPBModelData, save_model
)

setup_logging()
FIGS_DIR = "../waimlap2026/epb/figs"
os.makedirs(FIGS_DIR, exist_ok=True)


In [2]:
# ========== CONFIGURATION ==========
DATA_FOLDER = "data/all-images"         # Folder with all labeled images
LABEL_FILE = "data/epb_labels.csv"      # CSV with ground-truth labels and metadata

TARGET_SIZE = (64, 64)                  # Downsampled image resolution
AUGMENT_FACTOR = 2                      # Stratified augmentation factor (applied strictly inside training splits)
BALANCE_METHOD = "smote"                # In-fold SMOTE balancing

N_COMPONENTS = 5                        # 2DPCA components (64 x 5 = 320 features)
ENSEMBLE_PARAMS = {
    "n_components": 5,
    "svm_C": 5.0,
    "svm_gamma": "scale",
    "xgb_n_estimators": 200,
    "xgb_max_depth": 6,
    "xgb_learning_rate": 0.1,
    "rf_n_estimators": 200,
    "rf_max_depth": 15,
    "meta_C": 1.0,
}


In [3]:
MODEL_VERSION = "4.0"
np.random.seed(42)

print("="*70)
print("🌌 EPB DETECTION SYSTEM — V4 (COMPLETE BENCHMARK & EVALUATION)")
print("="*70)

df_labels = pd.read_csv(LABEL_FILE)

# 1. Standard Physics-Informed Preprocessed Dataset
X, y, filenames = load_epb_dataset(
    data_folder=DATA_FOLDER,
    label_file=LABEL_FILE,
    target_size=TARGET_SIZE,
    apply_preprocessing=True,
    apply_circular_mask=False
)

# 2. Raw Dataset (without CLAHE/Gaussian/Star-removal) for Ablation Study
X_raw, _, _ = load_epb_dataset(
    data_folder=DATA_FOLDER,
    label_file=LABEL_FILE,
    target_size=TARGET_SIZE,
    apply_preprocessing=False,
    apply_circular_mask=False
)

# 3. Circular FOV Masked Dataset (removes corner text overlays) for Confounder Ablation
X_masked, _, _ = load_epb_dataset(
    data_folder=DATA_FOLDER,
    label_file=LABEL_FILE,
    target_size=TARGET_SIZE,
    apply_preprocessing=True,
    apply_circular_mask=True
)

# Stratified 75/25 Split (preserving indices for all dataset variants)
indices = np.arange(len(X))
train_idx, test_idx = train_test_split(
    indices, test_size=0.25, random_state=42, stratify=y
)

X_train_orig, X_test = X[train_idx], X[test_idx]
y_train_orig, y_test = y[train_idx], y[test_idx]
files_train = [filenames[i] for i in train_idx]
files_test = [filenames[i] for i in test_idx]

X_train_raw, X_test_raw = X_raw[train_idx], X_raw[test_idx]
X_train_masked, X_test_masked = X_masked[train_idx], X_masked[test_idx]

print(f"\n📦 Stratified Holdout Split (75%/25%):")
print(f"   • Train (original): {len(X_train_orig)} images (EPB: {int(y_train_orig.sum())}, No EPB: {int(len(y_train_orig) - y_train_orig.sum())})")
print(f"   • Test  (original): {len(X_test)} images (EPB: {int(y_test.sum())}, No EPB: {int(len(y_test) - y_test.sum())})")


INFO: Carregando dataset de EPBs...


INFO: Labels carregados: 572 registros


🌌 EPB DETECTION SYSTEM — V4 (COMPLETE BENCHMARK & EVALUATION)


INFO: 572 imagens carregadas com sucesso


INFO: Dimensão das imagens: 64x64


INFO: Distribuição das classes:


... (13 outputs left out)

INFO: Distribuição das classes:


INFO:   EPBs detectados: 255 (44.6%)


INFO:   No EPB: 317 (55.4%)



📦 Stratified Holdout Split (75%/25%):
   • Train (original): 429 images (EPB: 191, No EPB: 238)
   • Test  (original): 143 images (EPB: 64, No EPB: 79)


In [4]:
# ========== PIPELINE ARCHITECTURE DIAGRAM ==========
fig_arch = plot_pipeline_architecture(
    save_path=f"{FIGS_DIR}/pipeline_architecture.png",
    dpi=300
)


💾 Pipeline architecture diagram saved: ../waimlap2026/epb/figs/pipeline_architecture.png (300 DPI)


/home/ivo/Documentos/Mestrado/dissertação/code/epb_detection_system_v2.py:4056: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ========== 2DPCA COMPONENT SELECTION & FULL EIGENVALUE SPECTRUM ==========
best_k, cv_comp_results = optimize_n_components(
    X_train_orig, y_train_orig,
    max_components=30,
    cv_folds=5,
    use_augmentation=False
)

# Augment training split only
X_train, y_train = augment_epb_data(X_train_orig, y_train_orig, augment_factor=AUGMENT_FACTOR)

# Fit full system
system = EPBRecognitionSystem(
    n_components=N_COMPONENTS,
    balance_data=True,
    balance_method=BALANCE_METHOD,
    ensemble_params=ENSEMBLE_PARAMS,
)
system.fit(X_train, y_train)

# Plot full 30-component eigenvalue spectrum + 5-Fold CV F1 curve
plot_2dpca_variance(
    system,
    cv_component_results=cv_comp_results,
    max_display_components=30,
    save_path=f"{FIGS_DIR}/2dpca_variance.png",
    dpi=300
)


INFO: Otimizando número de componentes (máx=30)...


INFO:    1 comp: F1=0.9552±0.0184 | Var=22.86%


INFO:    2 comp: F1=0.9733±0.0150 | Var=37.99%


INFO:    3 comp: F1=0.9708±0.0135 | Var=46.27%


INFO:    5 comp: F1=0.9663±0.0135 | Var=59.41%


INFO:   10 comp: F1=0.9714±0.0125 | Var=79.03%


... (30 outputs left out)

INFO: Classe 1: 573 → 573


INFO: Treinamento concluído!


💾 Figure saved: ../waimlap2026/epb/figs/2dpca_variance.png (300 DPI)


<Figure size 1300x450 with 3 Axes>

In [6]:
# ========== HOLDOUT TEST SET EVALUATION (n=143) ==========
print("="*70)
print("HOLDOUT TEST SET EVALUATION (n=143)")
print("="*70)

y_pred = system.predict(X_test)
y_proba = system.predict_proba(X_test)

print(classification_report(y_test, y_pred, target_names=["No EPB", "EPB"], digits=4))

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0

print(f"   • Sensitivity (Recall EPB):  {sensitivity*100:.2f}%")
print(f"   • Specificity:               {specificity*100:.2f}%")
print(f"   • PPV (Precision EPB):       {ppv*100:.2f}%")
print(f"   • NPV:                       {npv*100:.2f}%")


HOLDOUT TEST SET EVALUATION (n=143)
              precision    recall  f1-score   support

      No EPB     0.9625    0.9747    0.9686        79
         EPB     0.9683    0.9531    0.9606        64

    accuracy                         0.9650       143
   macro avg     0.9654    0.9639    0.9646       143
weighted avg     0.9651    0.9650    0.9650       143

   • Sensitivity (Recall EPB):  95.31%
   • Specificity:               97.47%
   • PPV (Precision EPB):       96.83%
   • NPV:                       96.25%


In [7]:
# ========== THRESHOLD SENSITIVITY & PROBABILITY CALIBRATION ==========
fig_threshold, df_thresholds = analyze_threshold_sensitivity(y_test, y_proba)

X_train_feat = system.scaler.transform(system.extract_features(X_train))
X_test_feat = system.scaler.transform(system.extract_features(X_test))

fig_cal, cal_stats = plot_calibration_analysis(
    y_true=y_test,
    y_proba=y_proba,
    system=system,
    X_train_features=X_train_feat,
    y_train=y_train,
    X_test_features=X_test_feat,
)


/home/ivo/Documentos/Mestrado/dissertação/code/epb_detection_system_v2.py:2522: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



📊 ANÁLISE DE SENSIBILIDADE AO THRESHOLD

   Metric                   t=0.50      Best F1  Equilibrium
   --------------------------------------------------------
   Threshold                  0.50         0.08         0.15
   Sensitivity              0.9531       1.0000       0.9688
   Specificity              0.9747       0.9620       0.9747
   Precision                0.9683       0.9552       0.9688
   F1-Score                 0.9606       0.9771       0.9688
   Accuracy                 0.9650       0.9790       0.9720

   💡 Recomendação:
      O melhor F1 ocorre em t=0.08 (< 0.50) — o threshold padrão
      é conservador. Baixar o threshold aumenta a detecção de EPBs.




📊 ANÁLISE DE CALIBRAÇÃO DE PROBABILIDADES

   ECE original (Expected Calibration Error): 0.0212
   ✓ Probabilidades bem calibradas
   ECE após calibração isotónica:              0.0172
   ⚠️  Calibração não trouxe melhoria significativa (Δ=0.0040)

   💡 Interpretação do reliability diagram:
      Se os pontos ficam ACIMA da diagonal: modelo é subconfiante
        (diz 60%, mas na verdade 80% são positivos)
      Se os pontos ficam ABAIXO da diagonal: modelo é sobreconfiante
        (diz 80%, mas na verdade só 60% são positivos)



/home/ivo/Documentos/Mestrado/dissertação/code/epb_detection_system_v2.py:2673: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ========== 5-FOLD STRATIFIED CV & TEMPORAL-BLOCK GROUP CV (N=572) ==========
# 1. Standard 5-Fold Stratified Cross-Validation (Leakage-Free)
fig_kfold, df_kfold, oof_strat = evaluate_kfold_cv(
    X, y,
    n_components=N_COMPONENTS,
    n_folds=5,
    balance_method=BALANCE_METHOD,
    use_augmentation=False,
    ensemble_params=ENSEMBLE_PARAMS,
    group_by=None,
    return_oof=True,
    save_path=f"{FIGS_DIR}/kfold_cv_metrics.png",
    dpi=300
)

# Save Out-of-Fold ROC, PR, and Probability Distribution curves (N=572)
plot_performance_metrics(
    oof_strat["y_true"], oof_strat["y_proba"],
    save_path=f"{FIGS_DIR}/roc_pr_curves.png",
    dpi=300
)

# Save Out-of-Fold Confusion Matrix (N=572)
plot_confusion_matrix_analysis(
    oof_strat["y_true"], oof_strat["y_pred"],
    save_path=f"{FIGS_DIR}/confusion_matrix.png",
    dpi=300
)

# 2. 5-Fold Temporal-Block Group CV (77 non-overlapping 30-min observation blocks)
fig_tg_cv, df_tg_cv = evaluate_kfold_cv(
    X, y,
    n_components=N_COMPONENTS,
    n_folds=5,
    balance_method=BALANCE_METHOD,
    use_augmentation=False,
    ensemble_params=ENSEMBLE_PARAMS,
    group_by="temporal_block",
    df_metadata=df_labels,
    dpi=300
)

# 3. 5-Fold Circular FOV Mask Ablation CV (verifying corner text stamps do not drive accuracy)
fig_mask_cv, df_mask_cv = evaluate_kfold_cv(
    X_masked, y,
    n_components=N_COMPONENTS,
    n_folds=5,
    balance_method=BALANCE_METHOD,
    use_augmentation=False,
    ensemble_params=ENSEMBLE_PARAMS,
    group_by=None,
    dpi=300
)


INFO: Iniciando treinamento do sistema...


INFO: Treinando 2DPCA para extração de features...


INFO: Variância explicada: 59.74%


INFO: Extraindo features...


INFO: Shape das features: (457, 320)


INFO: Normalizando features...


... (325 outputs left out)

   Fold 4/5 (n=114): Acc=0.9386 | Sens=0.8824 | Spec=0.9841 | Prec=0.9783 | F1=0.9278 | ROC-AUC=0.9975 | PR-AUC=0.9969


INFO: Treinamento concluído!


   Fold 5/5 (n=114): Acc=0.9561 | Sens=0.9216 | Spec=0.9841 | Prec=0.9792 | F1=0.9495 | ROC-AUC=0.9963 | PR-AUC=0.9957

📈 SUMMARY RESULTS: 5-Fold Stratified CV

   Metric                     Mean      ± Std        Min        Max
   ------------------------------------------------------------
   Accuracy                 0.9580     0.0157     0.9386     0.9826
   Sensitivity              0.9334     0.0472     0.8824     1.0000
   Specificity              0.9779     0.0180     0.9524     1.0000
   Precision                0.9724     0.0215     0.9423     1.0000
   F1                       0.9516     0.0189     0.9278     0.9808
   AUC                      0.9976     0.0011     0.9963     0.9991
   PR_AUC                   0.9971     0.0012     0.9957     0.9988


/home/ivo/Documentos/Mestrado/dissertação/code/epb_detection_system_v2.py:2892: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ========== BASELINE BENCHMARKING, ABLATION & CROSS-STATION GENERALIZATION ==========
fig_bench, df_bench = benchmark_baselines(
    X_train, y_train,
    X_test, y_test,
    n_components=N_COMPONENTS,
    balance_method=BALANCE_METHOD,
    ensemble_params=ENSEMBLE_PARAMS,
    X_train_raw=X_train_raw,
    X_test_raw=X_test_raw,
    X_train_masked=X_train_masked,
    X_test_masked=X_test_masked,
    y_train_orig=y_train_orig,
    X_all=X,
    y_all=y,
    X_all_raw=X_raw,
    X_all_masked=X_masked,
    df_metadata=df_labels,
    include_cnn=True,
    save_path=f"{FIGS_DIR}/benchmark_baselines.png",
    dpi=300
)



📊 COMPREHENSIVE BENCHMARK, ABLATION & CROSS-STATION GENERALIZATION STUDY
   Train: 1049 | Holdout Test: 143 | 2DPCA Components: 5



   Flatten + LogReg                   Acc=0.9790 | Sens=0.9688 | Spec=0.9873 | F1=0.9764 | AUC=0.9992 | ZS-CP=0.9762 | 0.050 ms/img
   1D-PCA + LogReg                    Acc=0.9860 | Sens=1.0000 | Spec=0.9747 | F1=0.9846 | AUC=0.9994 | ZS-CP=0.6905 | 0.070 ms/img


   1D-PCA + SVM (RBF)                 Acc=0.9510 | Sens=1.0000 | Spec=0.9114 | F1=0.9481 | AUC=0.9921 | ZS-CP=1.0000 | 0.231 ms/img


   Flatten + Random Forest            Acc=0.9720 | Sens=1.0000 | Spec=0.9494 | F1=0.9697 | AUC=0.9996 | ZS-CP=0.5476 | 0.678 ms/img


   2DPCA + SVM (RBF)                  Acc=0.9650 | Sens=0.9688 | Spec=0.9620 | F1=0.9612 | AUC=0.9978 | ZS-CP=1.0000 | 0.087 ms/img


   2DPCA + Random Forest              Acc=0.9371 | Sens=0.9062 | Spec=0.9620 | F1=0.9280 | AUC=0.9909 | ZS-CP=1.0000 | 0.749 ms/img


... (166 outputs left out)

INFO: Treinamento concluído!


   2DPCA + Stacking Ensemble ★        Acc=0.9650 | Sens=0.9531 | Spec=0.9747 | F1=0.9606 | AUC=0.9974 | ZS-CP=1.0000 | 0.630 ms/img

Model / Configuration                  Acc    Sens    Spec    Prec      F1     AUC   ZS-CP   ms/img
------------------------------------------------------------------------------------------------------
Flatten + LogReg                    0.9790  0.9688  0.9873  0.9841  0.9764  0.9992  0.9762    0.050
1D-PCA + LogReg                     0.9860  1.0000  0.9747  0.9697  0.9846  0.9994  0.6905    0.070
1D-PCA + SVM (RBF)                  0.9510  1.0000  0.9114  0.9014  0.9481  0.9921  1.0000    0.231
Flatten + Random Forest             0.9720  1.0000  0.9494  0.9412  0.9697  0.9996  0.5476    0.678
2DPCA + SVM (RBF)                   0.9650  0.9688  0.9620  0.9538  0.9612  0.9978  1.0000    0.087
2DPCA + Random Forest               0.9371  0.9062  0.9620  0.9508  0.9280  0.9909  1.0000    0.749
2DPCA + XGBoost                     0.9510  0.9375  0.9620  0.95

💾 Benchmark figure saved: ../waimlap2026/epb/figs/benchmark_baselines.png (300 DPI)


/home/ivo/Documentos/Mestrado/dissertação/code/epb_detection_system_v2.py:3374: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# ========== HARD NEGATIVES, LUNAR STRATIFICATION, CROSS-STATION & STAGE 1 QC ==========
gen_results = evaluate_hard_negatives_and_generalization(
    X=X,
    y=y,
    df_metadata=df_labels,
    oof_y_pred=oof_strat["y_pred"],
    oof_y_proba=oof_strat["y_proba"],
    system=system,
    util_dir="util",
    threshold_stars=4500,
)


INFO: Iniciando treinamento do sistema...


INFO: Treinando 2DPCA para extração de features...


INFO: Variância explicada: 59.81%


INFO: Extraindo features...


INFO: Shape das features: (530, 320)


INFO: Normalizando features...


... (35 outputs left out)

INFO: Classe 1: 255 → 255



2️⃣ Zero-Shot Cross-Station Generalization (Train: CA+BJL -> Test: CP):
   • Detected 42/42 EPB frames (100.00% Recall, Mean Confidence: 75.71%)

   Station Breakdown:
                    Station                      Evaluation_Mode   N  EPB  No_EPB  Accuracy
                         CA              5-Fold CV (Out-of-Fold) 310  213      97    0.9484
                        BJL              5-Fold CV (Out-of-Fold) 220    0     220    1.0000
                         CP              5-Fold CV (Out-of-Fold)  42   42       0    1.0000
CP (Unseen Station Holdout) Zero-Shot (Trained on CA+BJL, n=530)  42   42       0    1.0000


INFO: Treinamento concluído!



3️⃣ Stage 1 Sky Quality Control vs. Stage 2 Alone on Nocturnal Sequences:
                  Nocturnal_Sequence  Total_Frames QC_Flagged_Cloudy QC_Degraded Stage2_Only_EPB_Triggers Cascaded_Valid_EPB_Detections Cascaded_ClearSky_EPB
      BJL_20250722 (Clear Quiet Sky)           230         14 (6.1%)    9 (3.9%)                 3 (1.3%)                      0 (0.0%)              0 (0.0%)
CA_20260624 (Cloud-Contaminated Sky)            62        49 (79.0%)   7 (11.3%)               61 (98.4%)                    13 (21.0%)              6 (9.7%)
      CA_20251022 (Active EPB Night)           140          4 (2.9%)    5 (3.6%)              125 (89.3%)                   125 (89.3%)           122 (87.1%)


In [11]:
# ========== COMPREHENSIVE DASHBOARD & FEATURE INTERPRETABILITY ==========
plot_comprehensive_analysis(
    system, X_train, X_test, y_train, y_test,
    y_pred, y_proba, files_test
)

epb_correct_idx = np.where((y_test == 1) & (y_pred == 1))[0]
if len(epb_correct_idx) > 0:
    idx = epb_correct_idx[0]
    explain_feature_importance(system, X_test[idx], y_test[idx], files_test[idx], show_filename=False)
    explain_shap_importance(system, X_test[idx], y_test[idx], X_train, files_test[idx], show_filename=False)



🔍 FEATURE INTERPRETATION

📸 Imagem: O6_CA_20251023_014643.png
   • True class: EPB
   • Prediction: EPB
   • Probabilidade EPB: 99.15%
   • Confidence: Alta

📊 Most important regions (critical pixels):
   • Upper Left: 145 pixels (35.4%)
   • Lower Left: 140 pixels (34.1%)
   • Upper Right: 81 pixels (19.8%)
   • Lower Right: 44 pixels (10.7%)

💡 Interpretation:
   ✓ The model identified characteristic EPB patterns:
     - Vertical/irregular structures
     - Localized plasma depletions
     - Typical spatial distribution of bubbles


/home/ivo/Documentos/Mestrado/dissertação/code/epb_detection_system_v2.py:1746: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/home/ivo/miniconda3/envs/mestrado/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🔬 SHAP EXPLAINABILITY ANALYSIS — Supervised Pixel Attribution

📸 Image: O6_CA_20251023_014643.png
   • True class: EPB
   • Prediction: EPB (99.2%)
   • SHAP Method: TreeExplainer (XGBoost + RF)

📊 SHAP contribution by quadrant:
   • Upper Right: 40.4% (↑ EPB)
   • Upper Left: 37.1% (↑ EPB)
   • Lower Left: 13.2% (↑ EPB)
   • Lower Right: 9.4% (↑ EPB)

📐 Spearman Correlation (|SHAP| vs. 2DPCA Reconstruction Error): 0.532


/home/ivo/Documentos/Mestrado/dissertação/code/epb_detection_system_v2.py:4180: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# ========== SAVE TRAINED MODEL (V4.0) ==========
print("="*70)
print("SAVING TRAINED MODEL (V4.0)")
print("="*70)

model_data = EPBModelData(
    version=MODEL_VERSION,
    system=system,
    n_components=N_COMPONENTS,
    target_size=TARGET_SIZE,
    test_metrics={
        "accuracy": float(np.mean(y_pred == y_test)),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "ppv": float(ppv),
        "npv": float(npv),
        "cv_f1_mean": float(df_kfold["f1"].mean()),
        "cv_auc_mean": float(df_kfold["auc"].mean()),
    }
)

file_hash = save_model(model_data, f"models/epb_recognition_model-v{MODEL_VERSION}.pkl")
# Also update v3.0 pkl for backward compatibility
save_model(model_data, "models/epb_recognition_model-v3.0.pkl")

print(f"   ✓ Saved model: models/epb_recognition_model-v{MODEL_VERSION}.pkl (v{MODEL_VERSION})")
print(f"   ✓ SHA-256: {file_hash[:16]}...")
print("\n" + "="*70)
print("✅ V4 PIPELINE EXECUTED SUCCESSFULLY!")
print("="*70)


INFO: Modelo salvo: models/epb_recognition_model-v4.0.pkl (SHA-256: 5ce051d44907893c...)


SAVING TRAINED MODEL (V4.0)


INFO: Modelo salvo: models/epb_recognition_model-v3.0.pkl (SHA-256: 5ce051d44907893c...)


   ✓ Saved model: models/epb_recognition_model-v4.0.pkl (v4.0)
   ✓ SHA-256: 5ce051d44907893c...

✅ V4 PIPELINE EXECUTED SUCCESSFULLY!
